# Importation des bibliothèques

In [1]:
%%capture
## Instalation des bibliothèques utilitaires
!pip install google-api-python-client
!pip install --upgrade google-auth-httplib2 google-auth-oauthlib

In [2]:
%%capture

import logging
import warnings

warnings.filterwarnings("ignore")

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from openhexa.sdk import current_run, workspace

pd.set_option("display.max_columns", None)

try:
    import openpyxl as pyxl
    from fuzzywuzzy import fuzz, process
except ImportError or ModuleNotFoundError:
    !pip install python-Levenshtein
    !pip install fuzzywuzzy
    from fuzzywuzzy import fuzz, process
    import openpyxl as pyxl

import os
from functools import partial
from importlib import reload

In [3]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class LocalRun:
    """Mock current_run for local executions."""

    def log_info(self, msg: str) -> None:
        """Mock mock_run.log_info()."""
        logger.info(msg)

    def log_warning(self, msg: str) -> None:
        """Mock mock_run.log_warning()."""
        logger.warning(msg)

    def log_error(self, msg: str) -> None:
        """Mock mock_run.log_error()."""
        logger.error(msg)

    def add_file_output(self, fp: str) -> None:
        """Mock mock_run.add_file_output()."""
        logger.info(f"File output added: {fp}")


mock_run = current_run or LocalRun()

In [4]:
# Ajout du chemin d'accès aux différents code
os.chdir(Path(workspace.files_path, "Rapport Feedback/code/pipelines"))

# Module créer pour le processing et l'exportation des données
from compute_indicators import compute_indicators, date_utils, excel_file_handler
from database_operations import db_ops, update_dimension, upsert_table
from export_file_to_google_drive import upload_file_to_drive
from generate_feedback_report import generate_feedback_report as gfr
from metabase import queries
from metabase.metabase import Metabase

GraphQLClientGraphQLMultiError: Resolver requires an authenticated user

# Définition des paramètres

Les variables nécessaires à la production du rapport de feedback :

1. **Mois de création du rapport**  
   - **Description :** Ce paramètre correspond au mois pour lequel le rapport de feedback sera généré.  
   - **Source :** Défini par l'utilisateur lors de l'exécution du pipeline.

2. **Liste des sites attendus**  
   - **Description :** Fichier Excel contenant la liste complète des sites devant être inclus dans le rapport.  
   - **Fréquence de mise à jour :** Ce fichier est mis à jour et communiqué chaque trimestre.  
   - **Emplacement requis :** Le fichier doit être déposé dans le dossier `Rapport Feedback/data/Sites attendus`.

3. **Liste des produits traceurs**  
   - **Description :** Fichier Excel fournissant les informations sur les produits traceurs à analyser dans le rapport.  
   - **Fréquence de mise à jour :** Généralement communiqué mensuellement par la DAP.  
   - **Emplacement requis :** Le fichier doit être placé dans le dossier `Rapport Feedback/data/Produits Traceurs`.

<div class="alert alert-block alert-success">
Ici quand le mois est défini il faudrait aller checker dans la base de données pour voir si le mois en question est déjà présent dans la base pour minimiser le risque d'erreur
</div>

In [ ]:
month_report, fp_site_attendus, fp_prod_traceurs = (
    "Mars",
    "Sites attendus Février 2025.xlsx",
    "Liste des Produits Traceurs Février 2025.xlsx",
)

In [ ]:
month_export, date_report = (
    month_report,
    compute_indicators.generate_month_end_report_date(month_report),
)

## Test pour s'assurer que cette date n'existe pas déjà dans la base de données

In [ ]:
db_ops.reload_connection()

schema_name = "dap_tools"

In [ ]:
# Ici il faudrait également s'assurer que le mois précédent est bien présent à l'intérieur de la base de données au cas où on arriverait à se tromper sur le mois de conception du fichier
mois_prec = (pd.to_datetime(date_report).replace(day=1) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

df_ = pd.read_sql(
    f"select * from {schema_name}.etat_de_stock where date_report='{mois_prec}'", db_ops.civ_engine
)

assert df_.shape[0] != 0, (
    f"Le mois précédent {mois_prec} n'est pas présent dans la base de données locale êtes-vous sûre d'avoir choisir le bon mois de conception du fichier."
)

del df_

In [ ]:
%%script false --no-raise-error

df_ = pd.read_sql(
    f"select * from {schema_name}.etat_de_stock where date_report='{pd.to_datetime(date_report).strftime('%Y-%m-%d')}' LIMIT 2",
    db_ops.civ_engine,
)

assert df_.shape[0] == 0, (
    f"La date {date_report} existe déjà dans la base de données vous devez définir une autre date pour la production du rapport."
)

del df_

# 📥 Importation des Données
  
L'utilisateur doit veiller à ce que les fichiers respectent le format attendu et soient placés dans les répertoires dédiés avant de procéder au traitement.

## 📌 Importation de la `Liste des sites attendus`
- **Format requis :** Assurez-vous que le fichier respecte le template standard défini pour les mois de chargement.
- **Emplacement du fichier :** Le fichier doit être placé dans le répertoire dédié :  
  **`Rapport Feedback/data/Sites attendus`**  
- **En cas d'erreur :**  
  - Vérifiez que le fichier est bien présent dans le répertoire.  
  - Assurez-vous que toutes les colonnes requises sont bien renseignées.  
  - Contrôlez que le fichier est bien accessible et non corrompu.

In [ ]:
fp_site_attendus = Path(fp_site_attendus)

# (
#    Path(workspace.files_path)
#    / "Rapport Feedback/data/Sites attendus"
#    / Path(fp_site_attendus).name
# )

In [ ]:
# Configuration constante
EXPECTED_COLS = {
    "Code",
    "Site",
    "District",
    "Region",
    "ARV",
    "TRC",
    "LAB",
    "CHARGE VIRALE",
    "PNLP",
    "PNSME",
    "PNSME-GRAT",
    "PNN",
    "TBS",
    "TBMR",
    "TBLAB",
}

# Utilisation
try:
    df_site_attendu = excel_file_handler.load_expected_sites_from_excel(
        fp_site_attendus, EXPECTED_COLS
    )
except Exception as e:
    mock_run.log_error(
        "Une erreur s'est produite lors du chargement du fichier contenant la liste des sites attendus."
    )
    mock_run.log_error(
        "Veuillez vérifier que le fichier existe dans le répertoire spécifique `Rapport Feedback/data/Sites attendus` et qu'il contient toutes les colonnes requises."
    )
    mock_run.log_error(f"Code d'erreur détaillé : {e}")
    raise

df_site_attendu.head(2)

## 📌 Importation de la `liste des produits traceurs`
- **Format requis :** Respectez la structure du fichier définie pour l’importation.  
- **Emplacement du fichier :** Le fichier doit être placé dans le répertoire suivant :  
  **`Rapport Feedback/data/Produits Traceurs`**  
- **En cas d'erreur :**  
  - Vérifiez que le fichier est bien présent dans le répertoire.  
  - Assurez-vous que les données respectent le format attendu.  
  - Vérifiez l'intégrité du fichier et assurez-vous et corriger l'anomalie.

In [ ]:
# Il faudra définir un standard de fichier qui sera attendu
fp_prod_traceurs = Path(fp_prod_traceurs)

# (
#    Path(workspace.files_path)
#    / "Rapport Feedback/data/Produits Traceurs"
#    / Path(fp_prod_traceurs).name
# )

In [ ]:
try:
    df_prod_traceurs = excel_file_handler.load_traceable_products_from_excel(fp_prod_traceurs)
except Exeption as e:
    mock_run.log_error(
        "Une erreur s'est produite lors du chargement du fichier contenant la liste des produits traceurs"
    )
    mock_run.log_error(
        "Veuillez vérifier que le fichier existe et qu'il contienu dans le répertoire dédié `Rapport Feedback/data/Produits Traceurs`"
    )
    mock_run.log_error(f"Code d'erreur détaillé : {e}")

df_prod_traceurs.head(2)

## 📌 Extraction des données eSIGL : `État de transmission`  

Cette étape permet d'extraire les données d'état de transmission depuis eSIGL via Metabase.  
Assurez-vous que toutes les conditions nécessaires sont remplies avant de lancer l'extraction.  

🛠 En cas d'erreur :  
1. **Vérifier les permissions utilisateur**  
   - Assurez-vous que l'utilisateur **`secretariat_dap`** dispose bien des accès nécessaires à eSIGL.  
   - Si l'accès est restreint, contactez l'administrateur du système pour obtenir les autorisations requises.  

2. **S'assurer du bon fonctionnement de eSIGL**  
   - Il arrive que **Metabase** subisse des interruptions temporaires rendant l'accès aux données impossible.  
   - Si eSIGL ou Metabase est indisponible, réessayez plus tard ou consultez l'équipe technique.  

**Astuce :** Si le problème persiste, tentez d'accéder manuellement à eSIGL et Metabase pour vérifier leur disponibilité.

In [ ]:
metabase = Metabase(workspace.custom_connection("metabase-esigl"))

In [ ]:
mock_run.log_info("Extraction des données de transmission depuis Metabase...")

try:
    df_transmission = metabase.get_data_from_sql_query(
        queries.QUERY_TRANSMISSION.format(date_report=date_utils.get_date_report(date_report))
    )

    # Les établissements ne sont plus censés faire des rapportages
    # sur ce programme spécifique
    df_transmission["program"] = df_transmission["program"].str.replace(
        "PNSME-MEDICAMENTS ET INTRANTS",
        "PNSME_GRATUITE:MEDICAMENTS ET INTRANTS",
    )

    mock_run.log_info(
        f"Données de transmission extraites avec succès ({len(df_transmission)} enregistrements)."
    )

except Exception as e:
    mock_run.log_error("Erreur lors de l'extraction des données de transmission depuis Metabase.")
    mock_run.log_error(f"Détail de l'erreur : {e}")
    raise

df_transmission.head(3)

## 📌 Extraction des données eSIGL : `Etat de stock`

In [ ]:
mock_run.log_info("Extraction des données d'état de stock depuis Metabase...")

try:
    df_etat_stock = metabase.get_data_from_sql_query(
        queries.QUERY_ETAT_STOCK.format(date_report=date_utils.get_date_report(date_report))
    )

    df_etat_stock["programme"] = df_etat_stock["programme"].str.replace(
        "PNSME-MEDICAMENTS ET INTRANTS",
        "PNSME_GRATUITE:MEDICAMENTS ET INTRANTS",
    )

    mock_run.log_info(
        f"Données d'état de stock extraites avec succès ({len(df_etat_stock)} enregistrements)."
    )
except Exception as e:
    mock_run.log_error.log_error(
        "Erreur lors de l'extraction des données d'Etat de Stock depuis Metabase."
    )
    mock_run.log_error(f"Détail de l'erreur : {e}")
    raise

df_etat_stock.head(3)

# 📊 Calcul des Indicateurs `(1/3)`

Cette étape consiste à **traiter les données du mois en cours** afin de calculer les indicateurs de performance, notamment :  
- **Complétude** : mesure le taux de soumission des données attendues.  
- **Promptitude** : évalue la rapidité avec laquelle les données sont transmises.
- **Etat de Stock**: informations sur les états de stock des différents produits.

Une fois calculés, ces indicateurs sont intégrés au **rapport de feedback**, qui sera ensuite exporté pour être accessible aux utilisateurs.

In [ ]:
%%time
mock_run.log_info("Calcul des indicateurs de complétude et de promptitude...")
df_ets, df_region = compute_indicators.compute_indicators_completeness_and_promptness(
    df_site_attendu.copy(), df_transmission.copy(), date_report
)

## 📄 Génération du Fichier Excel `(1/2)`

Cette étape permet de **générer un fichier Excel** contenant les indicateurs calculés et les données consolidées du rapport de feedback.  
Une fois le fichier créé, un **lien d’accès** est intégré dans le tableau de bord afin que les utilisateurs puissent le consulter et le télécharger facilement.  

In [ ]:
mock_run.log_info("Chargement du fichier du rapport Feedback")
src_file = (
    Path(workspace.files_path)
    / "Rapport Feedback/data/Template Rapport Feedback/RAPPORT FEEDBACK - TEMPLATE.xlsx"
)

wb_feedback_report = pyxl.load_workbook(src_file.as_posix())

In [ ]:
%%time
gfr.export_detail_comp_promp_to_sheet(
    wb_feedback_report, df_ets.copy(), df_region.copy(), date_report
)

## 📊 Suite du Calcul des Indicateurs `(2/3)`

Cette étape poursuit le traitement des données en complétant le calcul des indicateurs clés pour le rapport de feedback.  
Elle permet de finaliser les analyses et d’assurer la cohérence des résultats avant leur exportation.  

In [ ]:
%%time
(
    df_etat_stock,
    stock_lvl_decent,
    stock_region,
) = compute_indicators.analyze_product_stock_status_indicators(
    df_prod_traceurs.copy(), df_etat_stock.copy(), date_report
)

## 📄 Génération Finale du Fichier Excel `(2/2)`


In [ ]:
%%time
gfr.export_stock_data_to_sheet(wb_feedback_report, df_etat_stock.copy())

In [ ]:
%%time
# Date report doit être révue pour prendre le 11 du mois en cours
gfr.export_stock_region_to_sheet(
    wb_feedback_report,
    stock_lvl_decent.copy(),
    stock_region.copy(),
    date_report=pd.to_datetime(date_report).strftime("%Y/%m/%d"),
)

In [ ]:
# Sauvegarde du fichier dans un repertoire courant
dest_file = (
    Path(workspace.files_path)
    / f"Rapport Feedback/code/pipelines/rapport feedback genere/{date_report[:4]}"
)
dest_file.mkdir(exist_ok=True, parents=True)

dest_file = dest_file / f"Rapport FeedBack-{month_export.upper()}-{date_report[:4]}.xlsx"

wb_feedback_report.save(dest_file.as_posix())

del wb_feedback_report

## 📊 Suite du Calcul des Indicateurs `(3/3)`

In [ ]:
%%time
(
    stock_lvl_decent,
    stock_region,
    df_sheet_two,
    stock_region_with_central,
) = compute_indicators.aggregate_regional_stock_availability_metrics(
    stock_lvl_decent.copy(), stock_region.copy()
)

# Exportation des données vers la base des données

## Export du lien du fichier vers le répertoire drive

In [ ]:
mock_run.log_info("Exportation du rapport Feedback sur Google Drive")
share_link = upload_file_to_drive.upload_file_and_get_share_link(
    dest_file.as_posix(), date_report=date_report
)

df_share_link = pd.DataFrame(
    data=[{"share_link": share_link, "date_report": pd.to_datetime(date_report)}]
)

db_ops.civ_cursor.execute(f"""
DELETE FROM {schema_name}.share_link_fbr
WHERE date_report = '{date_report}'
""")

db_ops.conn.commit()

df_share_link.to_sql(
    "share_link_fbr",
    con=db_ops.civ_engine,
    schema=schema_name,
    index=False,
    if_exists="append",
)

db_ops.civ_engine.dispose()

del df_share_link

## Exportation des autres tables

In [ ]:
# completude_promptitude_par_ets
mock_run.log_info(
    "Préparation des DataFrames de complétude et de promptitude pour le chargement en base de données"
)
df_comp_promp_ets = df_ets.rename(
    columns=lambda x: (
        x.lower().replace(" ", "_").replace("-", "_") if x not in ("Code", "Site", "Region") else x
    )
)

# completude_promptitude_attendu_taux_region
df_comp_promp_region = df_region.rename(
    columns=lambda x: x.lower().replace(" ", "_") if x not in ("Region") else x
)

mock_run.log_info("Préparation des DataFrames de complétude et de promptitude terminée.")

In [ ]:
date_report = pd.to_datetime(date_report).strftime("%d-%m-%Y")

In [ ]:
df_etat_stock["date_report"] = pd.to_datetime(date_report, format="%d-%m-%Y")
stock_region_with_central["date_report"] = pd.to_datetime(date_report, format="%d-%m-%Y")
df_sheet_two["date_report"] = pd.to_datetime(date_report, format="%d-%m-%Y")

# Some change here
df_comp_promp_ets["date_report"] = pd.to_datetime(date_report, format="%d-%m-%Y")
df_comp_promp_region["date_report"] = pd.to_datetime(date_report, format="%d-%m-%Y")

In [ ]:
df_etat_stock.rename(
    columns={
        "CODE": "Code_produit",
        "PROGRAMME": "programme_abrv",
        "SOUS-PROGRAMME": "sous_programme",
        "PERIODE": "Periode",
        "REGION": "Region",
        "DISTRICT": "District",
        "CODE ETS": "Code_ets",
        "STRUCTURE": "Structure",
        "CATEGORIE PRODUIT": "cat_produit",
        "PRODUIT": "produit_designation",
        "UNITE DE RAPPORTAGE": "unit_rapportage",
        "STOCK INITIAL": "stock_initial",
        "QUANTITE RECUE": "qte_recue",
        "QUANTITE UTILISEE": "qte_utilisee",
        "PERTES ET AJUSTEMENT": "perte_ajust",
        "JOURS DE RUPTURE": "j_rupture",
        "SDU": "sdu",
        "CMM ESIGL": "cmm_esigl",
        "CMM gestionnaire": "cmm_gest",
        "QUANTITE PROPOSEE": "qte_prop",
        "QUANTITE COMMANDEE": "qte_cmde",
        "QUANTITE APPROUVEE": "qte_approuv",
        "MSD": "msd",
        "ETAT DU STOCK": "etat_stock",
        "BESOIN CMMMANDE URGENTE": "besoin_cmde_urg",
        "BESOIN TRANSFERT IN": "besoin_trsf_in",
        "QUANTITE A TRANSFERER OUT": "qte_trsf_out",
        "CATEGORIE_DU_PRODUIT": "cat_du_produit",
    },
    inplace=True,
)

stock_region_with_central.rename(
    columns={"Code": "Code_produit", "MSD": "msd", "STATUT": "Statut"}, inplace=True
)

### Mise à jour des dimensions

#### `Dimensions régions`


Cette étape permet de maintenir à jour la **dimension des régions** dans la base de données.  
Elle garantit que toutes les informations régionales sont alignées avec les mises à jour effectuées dans **eSIGL**

 🛠 Actions réalisées :  
1. **Mise à jour des régions existantes**  
   - Si des modifications ont été apportées aux régions dans **eSIGL**, elles sont répercutées dans notre base de données.  
   - Les informations régionales (ex. : noms, codes, identifiants) sont mises à jour pour rester cohérentes avec la source officielle.  

2. **Ajout des nouvelles régions**  
   - Si certaines régions ne sont pas encore présentes dans notre base, elles sont automatiquement ajoutées.  
   - Cela garantit que l’ensemble des régions récentes est bien pris en compte dans les analyses et traitements futurs.  
  

In [ ]:
mock_run.log_info(
    "Recherche des nouvelles régions et des modifications dans la dimension 'dim_region'..."
)

try:
    df_new_region = update_dimension.update_dimension_table(
        dimension_name="dim_region",
        source_dfs=[
            df_transmission[["region", "id_region_esigl"]].rename(columns={"region": "Region"}),
            df_etat_stock[["Region", "id_region_esigl"]],
        ],
        merge_on=["id_region_esigl"],
        change_columns=["Region"],
        code_generation=partial(
            update_dimension.region_code_generation,
            metabase=metabase,
        ),
        schema_name=schema_name,
    )

    mock_run.log_info(
        f"Analyse de la dimension 'dim_region' terminée avec succès "
        f"({len(df_new_region)} région(s) à insérer ou à mettre à jour)."
    )
except Exception as e:
    mock_run.log_error(f"Detail de l'erreur {e}")
    raise

df_new_region.head(3)

In [ ]:
mock_run.log_info("Mise à jour de la dimension 'dim_region' dans la base de données...")

try:
    upsert_table.upsert_table(
        df_new_region,
        "dim_region",
        schema_name,
        engine=db_ops.civ_engine,
    )

    mock_run.log_info("La dimension 'dim_region' a été mise à jour avec succès.")

except Exception as e:
    mock_run.log_error(
        "Erreur lors de la mise à jour de la dimension 'dim_region' dans la base de données."
    )
    mock_run.log_error(f"Détail de l'erreur : {e}")
    raise

del df_new_region

In [ ]:
df_region = db_ops.get_data_from_database("dim_region")

df_region.head(2)

#### `Dimension districts`

Mise à Jour et Ajout d'Informations Complémentaires  

Avant d'exécuter le traitement, il est **essentiel** de mettre à jour les informations si des modifications ont été effectuées dans **eSIGL**, notamment sur les districts.  

Étapes à suivre :  
1. **Vérification et mise à jour des informations existantes**  
   - Si des districts ont été modifiés ou ajoutés dans **eSIGL**, il faut s'assurer que ces changements sont bien répercutés dans les données utilisées.  

2. **Ajout des informations complémentaires**  
   - Une fois la mise à jour effectuée, il est nécessaire d'ajouter toutes les informations additionnelles requises pour le bon fonctionnement du processus.

In [ ]:
mock_run.log_info(
    "Recherche des nouveaux districts et des modifications dans la dimension 'dim_district'..."
)
df_new_district = update_dimension.update_dimension_table(
    dimension_name="dim_district",
    source_dfs=[
        df_transmission[["id_region_esigl", "district", "id_district_esigl"]].rename(
            columns={"district": "District"}
        ),
        df_etat_stock[["id_region_esigl", "District", "id_district_esigl"]],
    ],
    merge_on=["id_district_esigl"],
    change_columns=["District"],
    code_generation=partial(
        update_dimension.district_code_generation,
        tb_region="dim_region",
        schema_name=schema_name,
    ),
    schema_name=schema_name,
)
mock_run.log_info(
    f"Analyse de la dimension 'dim_district' terminée avec succès "
    f"({len(df_new_district)} district(s) à insérer ou à mettre à jour)."
)

In [ ]:
upsert_table.upsert_table(df_new_district, "dim_district", schema_name, engine=db_ops.civ_engine)

del df_new_district

In [ ]:
df_district_db = db_ops.get_data_from_database("dim_district")

#### `Dimensions Structure`

##### Première mise à jour de la dimensions structure avec les données eSIGL

In [ ]:
mock_run.log_info(
    "Recherche des nouvelles structures et des modifications dans la dimension 'dim_structure'..."
)
df_new_structure = update_dimension.update_dimension_table(
    dimension_name="dim_structure",
    source_dfs=[
        df_etat_stock[["Code_ets", "Structure", "TYPE DE STRUCTURE", "id_district_esigl"]].rename(
            columns={"TYPE DE STRUCTURE": "type_structure"}
        )
    ],
    merge_on=["Code_ets"],
    change_columns=["Structure", "type_structure"],
    schema_name=schema_name,
)
mock_run.log_info(
    f"Analyse de la dimension 'dim_structure' terminée avec succès "
    f"({len(df_new_structure)} structure(s) à insérer ou à mettre à jour)."
)

In [ ]:
# Exportation des données vers la BD
mock_run.log_info("Mise à jour de la dimension 'dim_structure' dans la base de données...")
upsert_table.upsert_table(df_new_structure, "dim_structure", schema_name, engine=db_ops.civ_engine)
mock_run.log_info("La dimension 'dim_structure' a été mise à jour avec succès.")

del df_new_structure

##### Seconde mise à jour avec la liste des sites attendus

In [ ]:
df_site_attendu["District_standard"] = df_site_attendu["District"].apply(
    lambda x: excel_file_handler.standardize_text(x)
)

df_district_db = db_ops.get_data_from_database("dim_district")
df_structure_db = db_ops.get_data_from_database("dim_structure")
df_structure_db["Code_ets"] = df_structure_db["Code_ets"].astype("Int64")

df_district_db["District"] = df_district_db["District"].apply(
    lambda x: excel_file_handler.standardize_text(x)
)

df_site_attendu = df_site_attendu.merge(
    df_district_db,
    left_on="District_standard",
    right_on="District",
    suffixes=("", "_existing"),
)

df_new_structure = (
    df_site_attendu[["Code", "Site", "id_district_esigl"]]
    .drop_duplicates()
    .rename(columns={"Code": "Code_ets", "Site": "Structure"})
)

df_new_structure = df_new_structure.loc[
    ~df_new_structure["Code_ets"].isin(df_structure_db["Code_ets"])
]

df_new_structure = df_new_structure.merge(df_district_db, on="id_district_esigl")
df_new_structure["type_structure"] = np.nan

df_new_structure = df_new_structure[df_structure_db.columns.to_list()]
mock_run.log_info(
    f"Préparation terminée : {len(df_new_structure)} nouvelle(s) structure(s) identifiée(s)."
)

In [ ]:
# Exportation des données vers la BD
upsert_table.upsert_table(df_new_structure, "dim_structure", schema_name, engine=db_ops.civ_engine)

del df_new_structure

In [ ]:
df_structure_db = db_ops.get_data_from_database("dim_structure")

#### `Dimensions programme`

In [ ]:
mock_run.log_info("Préparation de la dimension 'Programme'...")
prog_extract_stock = set(df_etat_stock.programme_abrv.unique()).union(["TOUS"])

df_programme = (
    pd.DataFrame({"Programme": list(prog_extract_stock)})
    .sort_values(by="Programme")
    .reset_index()
    .drop(columns="index")
)
df_programme.dropna(inplace=True)

del prog_extract_stock

# Pour l'instant ce n'est que les produits des 5 programmes de santé qui sont gérés
programme_order = {"PNLS": 1, "PNLP": 2, "PNSME": 3, "PNN": 4, "PNLT": 5, "TOUS": 6}
df_programme["programme_order"] = df_programme["Programme"].map(programme_order)
df_programme.sort_values("programme_order", inplace=True)
del programme_order

df_programme = db_ops.get_full_table(
    df_programme,
    "dim_programme",
)
mock_run.log_info(f"Dimension 'Programme' préparée avec succès ({len(df_programme)} programme(s)).")
df_programme

#### `Dimension sous_programme`

In [ ]:
df_sous_prog_db = db_ops.get_data_from_database(
    "dim_sous_programme",
)

df_sous_prog_db = df_sous_prog_db[["Programme", "Sous_programme"]].rename(
    columns={"Programme": "programme_abrv", "Sous_programme": "sous_programme"}
)

In [ ]:
mock_run.log_info("Préparation de la dimension 'Sous-programme'...")
df_sous_prog = pd.concat(
    [
        df_etat_stock[["programme_abrv", "sous_programme"]].drop_duplicates(),
        df_sous_prog_db,
    ],
    ignore_index=True,
).drop_duplicates()

df_sous_prog = (
    df_sous_prog.sort_values(["programme_abrv", "sous_programme"])
    .reset_index()
    .drop(columns="index")
)

# df_sous_prog['Code_sous_prog'] =
df_sous_prog["Occurence"] = df_sous_prog.groupby("programme_abrv").cumcount() + 1
df_sous_prog["Code_sous_prog"] = (
    df_sous_prog["programme_abrv"] + "-" + df_sous_prog["Occurence"].astype(str)
)

df_sous_prog = df_sous_prog[["Code_sous_prog", "sous_programme", "programme_abrv"]].rename(
    columns={"sous_programme": "Sous_programme", "programme_abrv": "Programme"}
)
df_sous_prog = df_sous_prog.map(lambda x: x.lstrip().rstrip() if isinstance(x, str) else x)

del df_sous_prog_db


In [ ]:
df_sous_prog = db_ops.get_full_table(
    df_sous_prog,
    "dim_sous_programme",
)

df_sous_prog.head(2)

#### `Dimension produit`

In [ ]:
mock_run.log_info("Préparation de la dimension 'Produit'...")
df_new_product = (
    df_etat_stock[
        [
            "Code_produit",
            "sous_programme",
            "produit_designation",
            "unit_rapportage",
            "cat_du_produit",
            "cat_produit",
        ]
    ]
    .drop_duplicates()
    .rename(columns={"sous_programme": "Sous_programme"})
)

df_new_product = df_new_product.map(lambda x: x.lstrip().rstrip() if isinstance(x, str) else x)

df_new_product = (
    df_new_product.merge(df_sous_prog, how="left", on="Sous_programme")
    .drop(columns=["Sous_programme", "Programme"])
    .rename(
        columns={
            "cat_du_produit": "Categorie_du_produit",
            "cat_produit": "Categorie_produit",
        }
    )
    .rename(columns=lambda x: x.capitalize())
    .sort_values("Code_produit")
    .reset_index(drop=True)
)
mock_run.log_info(
    f"Préparation de la dimension 'Produit' terminée ({len(df_new_product)} produit(s) identifiés)."
)

In [ ]:
mock_run.log_info(
    "Identification des nouveaux produits et des modifications de la dimension 'Produit'..."
)

df_new_product["Code_produit"] = df_new_product["Code_produit"].astype(str)

mock_run.log_info("Chargement de la dimension 'Produit' existante...")

df_product_db = db_ops.get_data_from_database("dim_produit")

df_new_product = df_new_product.merge(
    df_product_db,
    on=["Code_produit", "Code_sous_prog"],
    how="left",
    suffixes=("_new", "_past"),
)

condition = ""
for col in [col.replace("_past", "") for col in df_new_product.columns if "_past" in col]:
    condition += f"(df_new_product.{col}_new != df_new_product.{col}_past) | "

condition = condition.rstrip("| ")

df_new_product = df_new_product.loc[eval(condition)]

if not df_new_product.empty:
    df_new_product = df_new_product.drop_duplicates(
        subset=[
            "Code_produit",
            "Unit_rapportage_new",
            "Categorie_du_produit_new",
            "Categorie_produit_new",
            "Code_sous_prog",
            "id_produit_pk",
        ],
        keep="last",
    )

    max_val = df_product_db["id_produit_pk"].max() + 1
    missing = df_new_product["id_produit_pk"].isna()

    df_new_product.loc[missing, "id_produit_pk"] = range(
        max_val,
        max_val + missing.sum(),
    )

    df_new_product.columns = df_new_product.columns.str.replace("_new", "")

    df_new_product = df_new_product[df_product_db.columns.to_list()]

mock_run.log_info(
    f"Identification terminée : {len(df_new_product)} produit(s) à insérer ou à mettre à jour."
)

df_new_product.head(2)

In [ ]:
df_new_product = df_new_product.drop_duplicates(
    subset=[
        "Code_produit",
        "Categorie_produit",
        "Code_sous_prog",
    ]
)

In [ ]:
upsert_table.upsert_table(df_new_product, "dim_produit", schema_name, engine=db_ops.civ_engine)

del df_new_product

### Table de faits `Complétude` et `Promptitude`

In [ ]:
df_region = db_ops.get_data_from_database("dim_region")

#### `df_comp_promp_ets`

In [ ]:
db_ops.civ_cursor.execute(
    f"""
DELETE FROM {schema_name}.comp_promp_par_ets
WHERE date_report = '{pd.to_datetime(date_report, format="%d-%m-%Y").strftime("%Y-%m-%d")}'
"""
)

db_ops.conn.commit()

df_comp_promp_ets.drop(columns=["Site", "Region"]).rename(columns={"Code": "Code_ets"}).to_sql(
    "comp_promp_par_ets",
    con=db_ops.civ_engine,
    schema=schema_name,
    index=False,
    if_exists="append",
)

db_ops.civ_engine.dispose()

#### `df_comp_promp_region`

In [ ]:
db_ops.civ_cursor.execute(
    f"""
DELETE FROM {schema_name}.comp_promp_attendu_region
WHERE date_report = '{pd.to_datetime(date_report, format="%d-%m-%Y").strftime("%Y-%m-%d")}'
"""
)

db_ops.conn.commit()


df_comp_promp_region = df_comp_promp_region.map(
    lambda x: x.lstrip().rstrip() if isinstance(x, str) else x
)

df_comp_promp_region.merge(df_region[["Code_region", "Region"]], on="Region", how="left").drop(
    columns="Region"
).rename(columns=lambda x: x.replace("région", "region")).to_sql(
    "comp_promp_attendu_region",
    con=db_ops.civ_engine,
    schema=schema_name,
    index=False,
    if_exists="append",
)

db_ops.civ_engine.dispose()

### Table de faits Etat de stocks

#### `recap_stock_by_region`

In [ ]:
db_ops.civ_cursor.execute(
    f"""
DELETE FROM {schema_name}.recap_stock_by_region
WHERE date_report = '{pd.to_datetime(date_report, format="%d-%m-%Y").strftime("%Y-%m-%d")}'
"""
)

db_ops.conn.commit()

df_sheet_two.merge(df_region[["Code_region", "Region"]]).drop(columns="Region").to_sql(
    "recap_stock_by_region",
    con=db_ops.civ_engine,
    schema=schema_name,
    index=False,
    if_exists="append",
)

db_ops.civ_engine.dispose()

In [ ]:
# Get full data product
full_product = db_ops.get_data_from_database(
    "dim_produit",
)
full_product["Programme"] = full_product["Code_sous_prog"].apply(lambda x: x.split("-")[0])
full_product["Code_produit"] = full_product["Code_produit"].astype(str)
full_product.head(2)

#### `recap_stock_prog_region`

In [ ]:
db_ops.civ_cursor.execute(
    f"""
DELETE FROM {schema_name}.recap_stock_prog_region
WHERE date_report = '{pd.to_datetime(date_report, format="%d-%m-%Y").strftime("%Y-%m-%d")}'
"""
)

db_ops.conn.commit()

In [ ]:
stock_region["Code"] = stock_region["Code"].astype(str)
stock_region["date_report"] = pd.to_datetime(date_report, format="%d-%m-%Y")
stock_region = stock_region.map(lambda x: x.lstrip().rstrip() if isinstance(x, str) else x)
stock_region["MSD"] = stock_region["MSD"].apply(
    lambda x: str(round(float(x), 1)).replace(".", ",") if x != "NA" else "NA"
)

assert (
    stock_region.merge(
        full_product[["id_produit_pk", "Code_produit", "Programme"]].drop_duplicates(
            subset=["Code_produit", "Programme"]
        ),
        left_on=["Code", "Programme"],
        right_on=["Code_produit", "Programme"],
        how="left",
    )
    .merge(df_region[["Code_region", "Region"]], on="Region", how="left")
    .drop(columns=["Code", "Code_produit", "Region"])
    .shape[0]
    == stock_region.shape[0]
)

In [ ]:
stock_region.merge(
    full_product[["id_produit_pk", "Code_produit", "Programme"]].drop_duplicates(
        subset=["Code_produit", "Programme"]
    ),
    left_on=["Code", "Programme"],
    right_on=["Code_produit", "Programme"],
    how="left",
).merge(df_region[["Code_region", "Region"]], on="Region", how="left").drop(
    columns=["Code", "Code_produit", "Region"]
).rename(columns={"id_produit_pk": "id_produit_fk"}).to_sql(
    "recap_stock_prog_region",
    con=db_ops.civ_engine,
    schema=schema_name,
    index=False,
    if_exists="append",
)

db_ops.civ_engine.dispose()

#### `recap_stock_prog_nat`

In [ ]:
db_ops.civ_cursor.execute(
    f"""
DELETE FROM {schema_name}.recap_stock_prog_nat
WHERE date_report = '{pd.to_datetime(date_report, format="%d-%m-%Y").strftime("%Y-%m-%d")}'
"""
)

db_ops.conn.commit()

In [ ]:
stock_national = stock_lvl_decent[
    [
        "Code",
        "Programme",
        "Region",
        "lvl_decent_msd",
        "lvl_decent_statut",
        "lvl_decent_conso",
        "lvl_decent_sdu",
        "lvl_decent_cmm",
        "dispo_globale",
        "dispo_globale_cible",
        "dispo_traceur",
        "dispo_traceur_cible",
    ]
].rename(
    columns={
        "lvl_decent_msd": "MSD",
        "lvl_decent_statut": "STATUT",
        "lvl_decent_conso": "conso_lvl_national",
        "lvl_decent_sdu": "sdu_lvl_national",
        "lvl_decent_cmm": "cmm_lvl_national",
    }
)

In [ ]:
stock_national["Code"] = stock_national["Code"].astype(str)
stock_national = stock_national.map(lambda x: x.strip() if isinstance(x, str) else x)
stock_national["date_report"] = pd.to_datetime(date_report, format="%d-%m-%Y")

stock_national["MSD"] = stock_national["MSD"].apply(
    lambda x: str(round(float(x), 1)).replace(".", ",") if x != "NA" else "NA"
)

In [ ]:
assert (
    stock_national.merge(
        full_product[["id_produit_pk", "Code_produit", "Programme"]].drop_duplicates(
            subset=["Code_produit", "Programme"]
        ),
        left_on=["Code", "Programme"],
        right_on=["Code_produit", "Programme"],
        how="left",
    )
    .merge(df_region[["Code_region", "Region"]], on="Region", how="left")
    .drop(columns=["Code", "Code_produit", "Region"])
    .shape[0]
    == stock_national.shape[0]
)

stock_national = (
    stock_national.merge(
        full_product[["id_produit_pk", "Code_produit", "Programme"]].drop_duplicates(
            subset=["Code_produit", "Programme"]
        ),
        left_on=["Code", "Programme"],
        right_on=["Code_produit", "Programme"],
        how="left",
    )
    .merge(df_region[["Code_region", "Region"]], on="Region", how="left")
    .drop(columns=["Code", "Code_produit", "Region"])
    .rename(
        columns={
            "id_produit_pk": "id_produit_fk",
            "conso_lvl_national": "CONSO",
            "sdu_lvl_national": "SDU",
            "cmm_lvl_national": "CMM",
        }
    )
)

df_count_prog = stock_national.groupby(["Programme"])["id_produit_fk"].count().reset_index()

stock_national["statut_pourcentage"] = stock_national[["Programme", "STATUT"]].apply(
    lambda row: (
        1 / df_count_prog.loc[df_count_prog.Programme == row.Programme, "id_produit_fk"].iloc[0]
    ),
    axis=1,
)

stock_national.to_sql(
    "recap_stock_prog_nat",
    con=db_ops.civ_engine,
    schema=schema_name,
    index=False,
    if_exists="append",
)

db_ops.civ_engine.dispose()

del df_count_prog

#### `Etat de stock`

In [ ]:
mock_run.log_info(
    "Suppression des données existantes de la table 'etat_de_stock' pour la période de reporting..."
)
db_ops.civ_cursor.execute(
    f"""
DELETE FROM {schema_name}.etat_de_stock
WHERE date_report = '{pd.to_datetime(date_report, format="%d-%m-%Y").strftime("%Y-%m-%d")}'
"""
)

db_ops.conn.commit()

In [ ]:
mock_run.log_info("Chargement des données de l'état de stock du mois précédent...")

mois_prec = (
    pd.to_datetime(date_report, format="%d-%m-%Y").replace(day=1) - pd.Timedelta(days=1)
).strftime("%Y-%m-%d")

df_mois_prec = pd.read_sql(
    f"""
    SELECT *
    FROM {schema_name}.etat_de_stock
    WHERE date_report = '{mois_prec}'
    """,
    db_ops.civ_engine,
)

mock_run.log_info(
    f"Données du mois précédent chargées avec succès ({len(df_mois_prec)} enregistrement(s))."
)

In [ ]:
mock_run.log_info("Préparation des données d'état de stock avant leur chargement en base...")

# Nettoyage initial des données
df_etat_stock["Code_produit"] = df_etat_stock["Code_produit"].astype(str)
df_etat_stock = df_etat_stock.map(lambda x: x.strip() if isinstance(x, str) else x)

# Suppression des colonnes inutiles
cols_to_drop = ["programme_abrv", "Region", "District", "Structure"]
df_ = df_etat_stock.drop(columns=cols_to_drop)

# Association avec la dimension Sous-programme
df_ = df_.merge(
    df_sous_prog,
    left_on="sous_programme",
    right_on="Sous_programme",
    how="left",
).drop(columns=["Sous_programme", "Programme", "sous_programme"])

# Colonnes utilisées pour identifier un produit
dedup_cols = [
    "Code_produit",
    "Code_sous_prog",
    "Categorie_du_produit",
]

# Association avec la dimension Produit
df_ = df_.merge(
    full_product.drop_duplicates(subset=dedup_cols),
    right_on=dedup_cols,
    left_on=[
        "Code_produit",
        "Code_sous_prog",
        "cat_du_produit",
    ],
)

# Nettoyage final des colonnes
final_cols_to_drop = [
    "Code_produit",
    "cat_produit",
    "produit_designation",
    "unit_rapportage",
    "cat_du_produit",
    "Code_sous_prog",
    "Produit_designation",
    "Unit_rapportage",
    "Categorie_produit",
    "Categorie_du_produit",
    "Programme",
]

df_ = df_.drop(columns=final_cols_to_drop).rename(columns={"id_produit_pk": "id_produit_fk"})

mock_run.log_info(
    f"Préparation des données d'état de stock terminée ({len(df_)} enregistrement(s) prêts pour le chargement)."
)

In [ ]:
df_["Code_ets"] = df_["Code_ets"].astype("Int64")

In [ ]:
mock_run.log_info(
    "Contrôle de cohérence et enrichissement des données avec les informations du mois précédent..."
)

# Vérification que la jointure ne modifie pas le nombre d'enregistrements
assert (
    df_.merge(
        df_mois_prec[["Code_ets", "id_produit_fk", "qte_cmde", "etat_stock"]].rename(
            columns={
                "qte_cmde": "qte_cmde_mois_prec",
                "etat_stock": "etat_stock_mois_prec",
            }
        ),
        on=["Code_ets", "id_produit_fk"],
        how="left",
    ).shape[0]
    == df_.shape[0]
)

# Enrichissement avec les données du mois précédent
df_ = df_.merge(
    df_mois_prec[["Code_ets", "id_produit_fk", "qte_cmde", "etat_stock"]].rename(
        columns={
            "qte_cmde": "qte_cmde_mois_prec",
            "etat_stock": "etat_stock_mois_prec",
        }
    ),
    on=["Code_ets", "id_produit_fk"],
    how="left",
)

# Suppression des colonnes devenues inutiles
df_.drop(
    columns=[
        "id_region_esigl",
        "TYPE DE STRUCTURE",
        "id_district_esigl",
    ],
    inplace=True,
)

mock_run.log_info(
    "Contrôle de cohérence effectué et enrichissement avec les données du mois précédent terminé."
)

df_.head(3)

In [ ]:
df_.to_sql(
    "etat_de_stock",
    con=db_ops.civ_engine,
    schema=schema_name,
    index=False,
    if_exists="append",
)

db_ops.civ_engine.dispose()